In [1]:
import mediapipe as mp
import cv2
import numpy as np
import time
import threading

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "face_landmarker.task"

latest_result = None
result_lock = threading.Lock()

def store_result(result, output_image, timestamp_ms):
    global latest_result
    with result_lock:
        latest_result = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=store_result
)


In [ ]:
cap = cv2.VideoCapture(0)
start_time = time.time()

time.sleep(1)

with FaceLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int((time.time() - start_time) * 1000)
        landmarker.detect_async(mp_image, timestamp_ms)

        with result_lock:
            current_result = latest_result

        if current_result and current_result.face_landmarks:
            landmarks = current_result.face_landmarks[0]
            h, w, _ = frame.shape

            for lm in landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

    
            regions = {
                "forehead_ids": [109, 10, 338, 336, 9, 107],
                "left_cheek_ids": [116,111,117,118, 119, 120,100,142,36,205,123],
                "right_cheek_ids": [371,329,349,348,347,346,340,345,352,425,266]
            }

            for region_name, region in regions.items():
                points = np.array([
                    (int(landmarks[i].x * w), int(landmarks[i].y * h))
                    for i in region
                ])
                cv2.polylines(frame, [points], isClosed=True, color=(0,0,255) , thickness=2)



        cv2.imshow("rPPG", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1781246834.055329 17129853 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 12, prefix = pthread-default
W0000 00:00:1781246834.059128 17129853 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781246834.087771 17129853 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781246834.092504 17129855 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781246834.101246 17129855 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


KeyboardInterrupt: 